In [ ]:
#!/usr/bin/env python3
"""Loads the final results of one analysis for one MODEL: the summary of every cluster series (step 06), the joint cluster composition, the probe capture, the projections (step 05) and the species report tables.

ANALYSIS_NAME picks the analysis (None: the YAML analysis_name, else the newest analyses/*_SCambrian). In sc_summary.tsv every row is a kept sequence and every column ``{analysis}_{u|t|s}_{i|j}_{stat}`` a cluster series whose cells are SERIALs: ``<label>_001`` for internal clusters, letters for joint clusters, UPPER case when a joint cluster holds probe and target proteins. The species report tables (species_cluster_enrichment, species_top_clusters) belong to the h5ad_scanpy.report_series series.
"""
###################################### DEPENDENCIES_AND_CONSTANTS
import anndata as ad
from IPython.display import display
import pandas as pd
from pathlib import Path
from pyprojroot import find_root, has_file
import yaml
ANALYSIS_NAME = None
MODEL = "esmc_300m"
YAML_PATH = "configs/param_configs.yaml"
###################################### CONFIG
PROJECT_ROOT = find_root(has_file(".here"), start=Path.cwd()).resolve()
config = yaml.safe_load((PROJECT_ROOT / YAML_PATH).read_text()) or {}
pre = config["preprocess_sequences"]
analyses_dir = PROJECT_ROOT / pre["analyses_dir"]
analysis_dir = analyses_dir / (ANALYSIS_NAME or pre.get("analysis_name") or sorted(p.name for p in analyses_dir.glob("*_SCambrian") if p.is_dir())[-1])
run_dir = analysis_dir / MODEL
report_series = config.get("h5ad_scanpy", {}).get("report_series", "u_j_leiden2d")
print(f"[PYTHON-INFO] {run_dir.relative_to(PROJECT_ROOT)}: {sorted(p.name for p in run_dir.glob('*.tsv'))}")


In [ ]:
###################################### LOAD
summary = pd.read_csv(run_dir / "sc_summary.tsv", sep="\t", dtype=str)
joint_clusters = pd.read_csv(run_dir / "sc_joint_clusters.tsv", sep="\t", dtype={"serial": str})
probe_capture = pd.read_csv(run_dir / "sc_probe_capture.tsv", sep="\t")
adata = ad.read_h5ad(run_dir / "projections.h5ad")
report_paths = {"enrichment": run_dir / f"species_cluster_enrichment__{report_series}.csv", "top_clusters": run_dir / f"species_top_clusters__{report_series}.csv"}
report = {k: pd.read_csv(p, dtype={"cluster": str}) for k, p in report_paths.items() if p.exists()}
series = [c for c in summary.columns if c.startswith(f"{analysis_dir.name}_")]
print(f"[PYTHON-INFO] {len(summary):,} kept sequences, {len(series)} series, layouts {', '.join(adata.obsm.keys())}")
display(pd.DataFrame({"clusters": summary[series].nunique(), "unassigned": summary[series].isna().sum()}))
display(probe_capture.pivot(index="target", columns="series", values="pct_in_probe_clusters"))
